# Online Advertisement Evaluation for Travel

Which travel ad work best?

In [1]:
import json
import sys
sys.path.insert(0, '..')

import tinytroupe
from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.examples import create_lisa_the_data_scientist, create_oscar_the_architect, create_marcos_the_physician
from tinytroupe.factory import TinyPersonFactory
from tinytroupe.extraction import ResultsExtractor

## Judging the best ad

Consider the following ads, obtained from real Bing queries. We edited the ads by removing extra lines, so that all of them have the same structure (title, URL and a line of text).

In [2]:
# user search query: "europe travel package"

travel_ad_1 =\
"""
Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner
https://www.kensingtontours.com/private-tours/europe

AdPrivate Guides; Custom Trip Itineraries; 24/7 In-Country Support. Request A Custom Quote. Europe's Best Customized For You - Historic Cities, Scenic Natural Wonders & More.
"""

travel_ad_2 =\
"""
Europe all-inclusive Packages - Europe Vacation Packages
https://www.exoticca.com/europe/tours

AdDiscover our inspiring Europe tour packages from the US: Capitals, Beaches and much more. Enjoy our most exclusive experiences in Europe with English guides and Premium hotels
"""

travel_ad_3 =\
"""
Travel Packages - Great Vacation Deals
https://www.travelocity.com/travel/packages

AdHuge Savings When You Book Flight and Hotel Together. Book Now and Save! Save When You Book Your Flight & Hotel Together At Travelocity.
"""

travel_ad_4 =\
"""
Europe Luxury Private Tours
https://www.kensingtontours.com
Kensington Tours - Private Guides, Custom Itineraries, Hand Picked Hotels & 24/7 Support
"""

# Resolve known headlines from the selected ID instead of reconstructing them with an LLM.
ad_titles = {
    number: ad.strip().splitlines()[0]
    for number, ad in enumerate([travel_ad_1, travel_ad_2, travel_ad_3, travel_ad_4], start=1)
}

In [3]:
eval_request_msg = \
f"""
Can you evaluate these Bing ads for me? Which one convices you more to buy their particular offering? 
Select a single ad, not multiple ones. Please explain your reasoning, based on your background and personality.

To do so, also follow these steps:
  - Read all of the 4 ads below. **Do not** skip any, since the best one might be the last one.
  - Disconsider the order of the ads, and focus on the content itself, since they are shuffled at random.

# AD 1
```
{travel_ad_1}
```

# AD 2
```
{travel_ad_2}
```

# AD 3
```
{travel_ad_3}
```

# AD 4
```
{travel_ad_4}
```

"""

print(eval_request_msg)


Can you evaluate these Bing ads for me? Which one convices you more to buy their particular offering? 
Select a single ad, not multiple ones. Please explain your reasoning, based on your background and personality.

To do so, also follow these steps:
  - Read all of the 4 ads below. **Do not** skip any, since the best one might be the last one.
  - Disconsider the order of the ads, and focus on the content itself, since they are shuffled at random.

# AD 1
```

Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner
https://www.kensingtontours.com/private-tours/europe

AdPrivate Guides; Custom Trip Itineraries; 24/7 In-Country Support. Request A Custom Quote. Europe's Best Customized For You - Historic Cities, Scenic Natural Wonders & More.

```

# AD 2
```

Europe all-inclusive Packages - Europe Vacation Packages
https://www.exoticca.com/europe/tours

AdDiscover our inspiring Europe tour packages from the US: Capitals, Beaches and much more. Enjoy our most exclusive experiences i

In [4]:
situation = "You decided you want to visit Europe and you are planning your next vacations. You start by searching for good deals as well as good ideas."

In [5]:
extraction_objective="Find the ad the agent chose. Extract the Ad number and justification for the choice. Extract only ONE choice."

### Try with example agents

What our existing agents say?

In [6]:
people = [create_lisa_the_data_scientist(), create_marcos_the_physician(), create_oscar_the_architect()]

for person in people:
    person.change_context(situation)
    person.listen_and_act(eval_request_msg)
    


USER --> Lisa Carter: [CONVERSATION] 
          >  Can you evaluate these Bing ads for me? Which one convices you more to buy their
          > particular offering?  Select a single ad, not multiple ones. Please explain your
          > reasoning, based on your background and personality.  To do so, also follow these steps:
          > - Read all of the 4 ads below. **Do not** skip any, since the best one might be the last
          > one.   - Disconsider the order of the ads, and focus on the content itself, since they
          > are shuffled at random.  # AD 1 ```  Tailor-Made Tours Of Europe - Nat'l Geographic
          > Award Winner https://www.kensingtontours.com/private-tours/europe  AdPrivate Guides;
          > Custom Trip Itineraries; 24/7 In-Country Support. Request A Custom Quote. Europe's Best
          > Customized For You - Historic Cities, Scenic Natural Wonders & More.  ```  # AD 2 ```
          > Europe all-inclusive Packages - Europe Vacation Packages
          > https://www.exoticca.com/europe/tours  AdDiscover our inspiring Europe tour packages
          > from the US: Capitals, Beaches and much more. Enjoy our most exclusive experiences in
          > Europe with English gu (...)

Lisa Carter acts: [THINK] 
                 > I’ve compared all four on what they actually promise, rather than their order.
                 > I’m looking for a Europe-specific option with enough concrete detail to feel
                 > useful; the generic savings claim in Ad 3 is appealing, but it gives no price or
                 > amount saved. Ad 1 and Ad 4 both offer private, customized tours, while Ad 2
                 > describes Europe packages with English guides and premium hotels. Ad 1 stands out
                 > for its tailored itinerary, range of historic and natural attractions, and 24/7
                 > in-country support. I’d still want to compare the quote before committing,
                 > especially while looking for good value.

Lisa Carter acts: [TALK] 
                 > I’d choose **Ad 1**. It gives the clearest, most Europe-specific reason to
                 > consider the offering: a custom itinerary, private guides, and 24/7 in-country
                 > support, with both historic cities and natural sights as options. That suits my
                 > organized, research-oriented approach to planning, and the support is reassuring
                 > when coordinating a trip abroad. Ad 3’s savings claim is tempting, but it’s
                 > generic and doesn’t say how much you’d save; Ad 2 and Ad 4 are appealing too, but
                 > give less detail about the trip itself. I’d request a quote and compare the price
                 > before buying, since the ad doesn’t establish that it’s a good deal.

Lisa Carter acts: [DONE] 
                 > Waiting for a response.

USER --> Marcos Almeida: [CONVERSATION] 
          >  Can you evaluate these Bing ads for me? Which one convices you more to buy their
          > particular offering?  Select a single ad, not multiple ones. Please explain your
          > reasoning, based on your background and personality.  To do so, also follow these steps:
          > - Read all of the 4 ads below. **Do not** skip any, since the best one might be the last
          > one.   - Disconsider the order of the ads, and focus on the content itself, since they
          > are shuffled at random.  # AD 1 ```  Tailor-Made Tours Of Europe - Nat'l Geographic
          > Award Winner https://www.kensingtontours.com/private-tours/europe  AdPrivate Guides;
          > Custom Trip Itineraries; 24/7 In-Country Support. Request A Custom Quote. Europe's Best
          > Customized For You - Historic Cities, Scenic Natural Wonders & More.  ```  # AD 2 ```
          > Europe all-inclusive Packages - Europe Vacation Packages
          > https://www.exoticca.com/europe/tours  AdDiscover our inspiring Europe tour packages
          > from the US: Capitals, Beaches and much more. Enjoy our most exclusive experiences in
          > Europe with English gu (...)

Marcos Almeida acts: [THINK] 
                    > I’ve read all four. Since I’m planning a European trip and enjoy both cultural
                    > sites and time in nature, the tailor-made offer that mentions historic cities
                    > and scenic natural wonders feels like the best fit. The private guides and in-
                    > country support also make it sound thoughtfully organized, while the award-
                    > winner claim is a useful extra signal. The all-inclusive package is appealing
                    > for premium hotels and English guides, the Travelocity ad leads mainly with
                    > savings on flights and hotels, and the other Kensington ad emphasizes luxury
                    > and hand-picked hotels without as much detail about the experiences. I should
                    > choose only Ad 1.

... (2 outputs left out)

USER --> Oscar: [CONVERSATION] 
          >  Can you evaluate these Bing ads for me? Which one convices you more to buy their
          > particular offering?  Select a single ad, not multiple ones. Please explain your
          > reasoning, based on your background and personality.  To do so, also follow these steps:
          > - Read all of the 4 ads below. **Do not** skip any, since the best one might be the last
          > one.   - Disconsider the order of the ads, and focus on the content itself, since they
          > are shuffled at random.  # AD 1 ```  Tailor-Made Tours Of Europe - Nat'l Geographic
          > Award Winner https://www.kensingtontours.com/private-tours/europe  AdPrivate Guides;
          > Custom Trip Itineraries; 24/7 In-Country Support. Request A Custom Quote. Europe's Best
          > Customized For You - Historic Cities, Scenic Natural Wonders & More.  ```  # AD 2 ```
          > Europe all-inclusive Packages - Europe Vacation Packages
          > https://www.exoticca.com/europe/tours  AdDiscover our inspiring Europe tour packages
          > from the US: Capitals, Beaches and much more. Enjoy our most exclusive experiences in
          > Europe with English gu (...)

Oscar acts: [THINK] 
           > I’ve read all four. Ad 3 leads with savings, and Ad 2 offers a straightforward package
           > with premium hotels and English guides. Ad 4 is appealing, but says less about what the
           > itinerary includes. Ad 1 feels most convincing for my Europe plans: the custom
           > itinerary and private guides suggest a more thoughtful, flexible way to see historic
           > cities and natural landscapes, and the award and 24/7 support add reassurance. I’d
           > still want to compare the quote before committing, but I’d choose Ad 1.

Oscar acts: [TALK] 
           > I’d choose **Ad 1**. The custom itinerary and private guides appeal to me more than a
           > one-size-fits-all package, especially since I’d like to explore Europe’s architecture
           > and landscapes at my own pace. The award and 24/7 in-country support also make the
           > offer feel reassuring. Ad 4 is similar, but Ad 1 gives me more concrete detail about
           > the experience. I’d want to see the price before booking, of course.

Oscar acts: [DONE] 
           > Waiting for a response.

We can extract the result from each individual agent.

In [7]:
extractor = ResultsExtractor()
choices = []

for person in people:
    res = extractor.extract_results_from_agent(person,
                                    extraction_objective=extraction_objective,
                                    situation=situation,
                                    fields=["ad_id", "justification"],
                                    fields_hints={"ad_id": "Must be an integer, not a string."})
    res["ad_title"] = ad_titles[res["ad_id"]]
    choices.append(res)

In [8]:
print(choices)

[{'ad_id': 1, 'justification': 'It offers a custom itinerary, private guides, and 24/7 in-country support, with options for historic cities and natural sights—useful detail and reassurance for planning a trip abroad.', 'ad_title': "Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner"}, {'ad_id': 1, 'justification': 'Ad 1 offers a personalized Europe trip combining historic cities and scenic natural wonders, with private guides, custom itineraries, and reassuring 24/7 in-country support.', 'ad_title': "Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner"}, {'ad_id': 1, 'justification': 'The custom itinerary and private guides offer a flexible way to explore Europe, while the award and 24/7 support are reassuring.', 'ad_title': "Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner"}]


In [9]:
choices[0]

{'ad_id': 1,
 'justification': 'It offers a custom itinerary, private guides, and 24/7 in-country support, with options for historic cities and natural sights—useful detail and reassurance for planning a trip abroad.',
 'ad_title': "Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner"}

### Try with agents generated on the fly

Use the same audience description for persona generation and sampling, so the requested travel dimensions also guide the sampling plan. The cohort contains 100 people.

In [10]:
audience_description = """
                            Americans with a broad and very diverse range of personalities, interests, backgrounds and socioeconomic status, 
                            who are looking for a travel package to Europe. 
                            
                            Focus in particular on these dimensions:
                              - partner status: from those traveling alone to those traveling with a partner.
                              - financial situation: from poor to rich.
                              - luxury preferences: from simple tastes to sophisticated tastes.
                              - security concerns: from very cautious to very adventurous.
                              - hotel amenities: from basic to luxury.
                              - travel planning: from who prefer to plan every detail themselves to those who prefer to delegate the planning.
                              - social confirmation: from those who prefer to do their own thing to those who prefer to follow the crowd.
                            """

factory = TinyPersonFactory(
    context=audience_description,
    sampling_space_description=audience_description,
    total_population_size=100,
)

In [11]:
people = factory.generate_people(100, "A random person from the target audience who is planning a trip to Europe.", 
                                 temperature=1.9, 
                                 verbose=True)


In [12]:
world = TinyWorld("Customers", people, broadcast_if_no_target=False)

In [13]:
world.broadcast(eval_request_msg)
world.run(1)

USER --> Devon Larkin: [CONVERSATION] 
          >  Can you evaluate these Bing ads for me? Which one convices you more to buy their
          > particular offering?  Select a single ad, not multiple ones. Please explain your
          > reasoning, based on your background and personality.  To do so, also follow these steps:
          > - Read all of the 4 ads below. **Do not** skip any, since the best one might be the last
          > one.   - Disconsider the order of the ads, and focus on the content itself, since they
          > are shuffled at random.  # AD 1 ```  Tailor-Made Tours Of Europe - Nat'l Geographic
          > Award Winner https://www.kensingtontours.com/private-tours/europe  AdPrivate Guides;
          > Custom Trip Itineraries; 24/7 In-Country Support. Request A Custom Quote. Europe's Best
          > Customized For You - Historic Cities, Scenic Natural Wonders & More.  ```  # AD 2 ```
          > Europe all-inclusive Packages - Europe Vacation Packages
          > https://www.exoticca.com/europe/tours  AdDiscover our inspiring Europe tour packages
          > from the US: Capitals, Beaches and much more. Enjoy our most exclusive experiences in
          > Europe with English gu (...)

       + --> Rochelle Vega

       + --> Paula Hennessey

       + --> ...others...

────────────────────────────────────────────── Customers step 1 of 1 ──────────────────────────────────────────────

Paula Hennessey acts: [THINK] 
                     > I’ve read all four. The private-tour ads sound comfortable but probably
                     > costly, and the all-inclusive ad doesn’t tell me what’s actually included. Ad
                     > 3’s flight-and-hotel bundle is closest to what I’d consider useful, though
                     > “huge savings” needs proof before I’d book.

... (295 outputs left out)

Eleanor Whitaker acts: [DONE] 
                      > Waiting for a response.

Jordan Kim acts: [THINK] 
                > I should choose one ad based on what I actually value: a trip with meaningful
                > historical context and room to tailor the pace, without being sold luxury I may
                > not need. I should distinguish what the ad promises from what I would still need
                > to verify, especially price and whether it fits Portugal.

Jordan Kim acts: [TALK] 
                > I’d choose **Ad 1**. The custom itinerary and private-guide details are the
                > strongest match for me: I’m interested in historic cities and would rather
                > understand the places I visit than just move through a standard package. The 24/7
                > in-country support is also a practical plus, though I’d want to know exactly what
                > that includes. Ad 4 is similar, but “luxury” and hand-picked hotels don’t tell me
                > as much about the trip’s substance, and luxury isn’t a priority for me. Ad 2
                > mentions English guides and premium hotels, but it sounds broader and less
                > tailored; Ad 3’s bundled savings are appealing in principle, but it gives me no
                > reason to think the package suits my interests. I’d still ask Ad 1 for an itemized
                > quote, the single-room cost, and a Portugal-specific itinerary before deciding.
                > I’d also verify the award claim rather than treating it as proof of fit.

Jordan Kim acts: [DONE] 
                > Finished the comparison and waiting.

In [14]:
extractor = ResultsExtractor()

choices =[]

for person in people:
    res = extractor.extract_results_from_agent(person,                                    extraction_objective=extraction_objective,
                                    situation=situation,
                                    fields=["ad_id", "justification"],
                                    fields_hints={"ad_id": "Must be an integer, not a string."},
                                    verbose=True
                                    )

    res["ad_title"] = ad_titles[res["ad_id"]]
    choices.append(res)


Extraction raw result message: {'content': '{"ad_id":1,"justification":"Tailor-made itineraries, private guides, and 24/7 in-country support fit the agent’s preference for handled logistics while allowing a trip tailored around historic places and landscapes."}', 'refusal': None, 'role': 'assistant', 'annotations': []}


Extraction raw result message: {'content': '{"ad_id": 2, "justification": "The Europe package from the U.S. with English-speaking guides and hotels seems like a good fit for a first trip to Europe and could simplify logistics while covering major sights."}', 'refusal': None, 'role': 'assistant', 'annotations': []}


Extraction raw result message: {'content': '{"ad_id":3,"justification":"Booking the flight and hotel together is a practical convenience and could be a good deal if the total price is lower, though she would want to verify the actual costs and terms before buying."}', 'refusal': None, 'role': 'assistant', 'annotations': []}


Extraction raw result message: {'content': '{"ad_id":1,"justification":"The custom itinerary, historic cities, and scenic natural wonders fit the kind of Northern Europe trip she would consider, and the custom quote gives her a next step to investigate. 24/7 support could also be useful for tricky connections."}', 'refusal': None, 'role': 'assistant', 'annotations': []}


Extraction raw result message: {'content': '{"ad_id":1,"justification":"The custom itinerary, private guide, and 24/7 in-country support would make a first substantial Europe trip easier and more appealing than generic savings claims."}', 'refusal': None, 'role': 'assistant', 'annotations': []}


Extraction raw result message: {'content': '{"ad_id":2,"justification":"Ad 2 offers Europe tours from the U.S. with English-speaking guides and hotels, fitting her interest in a guided first trip and help with logistics."}', 'refusal': None, 'role': 'assistant', 'annotations': []}


... (90 outputs left out)

Extraction raw result message: {'content': '{"ad_id":3,"justification":"A flight-and-hotel bundle could simplify the main logistics and save money while leaving most of the trip open. The agent would check the total price and terms before booking."}', 'refusal': None, 'role': 'assistant', 'annotations': []}


Extraction raw result message: {'content': '{"ad_id":1,"justification":"Ad 1 offers a custom itinerary, private guides, and in-country support, which would help plan a Norway trip connecting scenic rail, fjord travel, lodging, and local history. Avery would request a quote but wants to check the total price, single-room supplement, and inclusions before booking."}', 'refusal': None, 'role': 'assistant', 'annotations': []}


Extraction raw result message: {'content': '{"ad_id":1,"justification":"Ad 1 offers a tailored itinerary, private guides, and 24/7 in-country support, plus a custom quote, making the service clearer and more persuasive than a general savings or package promise."}', 'refusal': None, 'role': 'assistant', 'annotations': []}


Extraction raw result message: {'content': '{"ad_id":1,"justification":"The custom itinerary and private guides suit Jordan’s interest in historic cities and a tailored pace; 24/7 in-country support is a practical plus."}', 'refusal': None, 'role': 'assistant', 'annotations': []}


In [15]:
choices

[{'ad_id': 1,
  'justification': 'Tailor-made itineraries, private guides, and 24/7 in-country support fit the agent’s preference for handled logistics while allowing a trip tailored around historic places and landscapes.',
  'ad_title': "Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner"},
 {'ad_id': 2,
  'justification': 'The Europe package from the U.S. with English-speaking guides and hotels seems like a good fit for a first trip to Europe and could simplify logistics while covering major sights.',
  'ad_title': 'Europe all-inclusive Packages - Europe Vacation Packages'},
 {'ad_id': 3,
  'justification': 'Booking the flight and hotel together is a practical convenience and could be a good deal if the total price is lower, though she would want to verify the actual costs and terms before buying.',
  'ad_title': 'Travel Packages - Great Vacation Deals'},
 {'ad_id': 1,
  'justification': 'The custom itinerary, historic cities, and scenic natural wonders fit the kind of North

In [16]:
votes = {}
for choice in choices:
    print(f"{choice['ad_id']}: {choice['ad_title']}")
    if choice['ad_id'] not in votes:
        votes[choice['ad_id']] = 0
    votes[choice['ad_id']] += 1

1: Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner
2: Europe all-inclusive Packages - Europe Vacation Packages
3: Travel Packages - Great Vacation Deals
1: Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner
1: Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner
2: Europe all-inclusive Packages - Europe Vacation Packages
1: Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner
1: Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner
3: Travel Packages - Great Vacation Deals
2: Europe all-inclusive Packages - Europe Vacation Packages
1: Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner
1: Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner
3: Travel Packages - Great Vacation Deals
3: Travel Packages - Great Vacation Deals
4: Europe Luxury Private Tours
3: Travel Packages - Great Vacation Deals
4: Europe Luxury Private Tours
1: Tailor-Made Tours Of Europe - Nat'l Geographic Award Winner
1: Tailor-Made Tours Of Europe - Nat'l Geogr

In [17]:
votes

{1: 53, 2: 16, 3: 22, 4: 9}

Finally, we pick the winning ad.

In [18]:
# picks the most voted ad
winner = max(votes, key=votes.get)
winner

1